In [13]:
%run langchain_common_notebook.py

In [14]:
docs = [
    "Hi there!",
    "Oh, hello!",
    "What's your name?",
    "My friends call me World",
    "Hello World! What is going on? Where have you been? I haven't seen you in a while. How have you been? I hope everything is going well for you. It's been a long time since we last spoke. Let's catch up soon and talk about what's new in our lives."
]

embeddings = databricks_embeddings.embed_documents(docs)

In [15]:
# This is just to show the embeddings in a nice table format. 
import pandas as pd
pd.DataFrame(embeddings)


,0,1,2,3,4,5,6,7,8,9,...,1014,1015,1016,1017,1018,1019,1020,1021,1022,1023
0,-0.616699,-0.956543,-0.299072,-0.097290,0.001817,-0.450928,-0.872559,0.469971,-0.143433,0.394531,...,-0.040070,-0.974121,0.251709,1.324219,1.075195,0.393799,0.635742,0.719238,-0.956543,-1.421875
1,-0.904297,-0.713867,-1.015625,-0.190308,-0.422363,0.028107,-0.603516,0.475098,-0.080261,-0.138428,...,0.181763,-0.880859,0.226196,1.352539,0.761230,0.298340,0.225220,0.347168,-0.400879,-1.791016
2,-0.205566,-0.922363,-1.655273,-0.558105,0.192383,-0.088013,-0.818359,0.455322,-0.683594,0.537109,...,0.636230,-0.285156,0.114868,-0.115051,0.316406,-0.200195,-1.322266,-0.771484,-0.880371,-0.860840
3,0.157837,-0.317871,-0.395996,-0.153687,-0.135132,-0.175171,-0.225342,0.501953,0.036133,0.064209,...,0.036133,-0.100159,0.773926,0.413574,0.159790,-0.024338,-1.099609,-0.122925,-0.382568,-0.977051
4,-0.480225,-0.048553,-0.376953,-0.226440,-0.762207,0.262939,0.082520,0.490234,-1.064453,0.322754,...,0.082397,-1.041992,0.218994,1.260742,0.000637,0.656250,0.416016,-0.533691,0.154175,-0.987305


### Text String Embedding (PGVector.from_texts)

In [16]:
# from_texts: loads text strings into the vector store, embedding them in the process. 
# This is the most common way to load data into a vector store.    
db = PGVector.from_texts(docs, 
                         databricks_embeddings, 
                         collection_name="test_sentences", 
                         connection=pgvectordb_conn,
                         use_jsonb=True)

In [17]:
# Similarity search with scores
# You can also get similarity search without scores using `db.similarity_search()`, which returns just the documents.
scored_results = db.similarity_search_with_score("Hi there!", k=3)

for i, (doc, score) in enumerate(scored_results, 1):
    print(f"{i}. score={score:.6f} | text={doc.page_content}")

1. score=0.000001 | text=Hi there!
2. score=0.000001 | text=Hi there!
3. score=0.000001 | text=Hi there!


In [18]:
# Retriever interface (RAG-friendly)
retriever = db.as_retriever(search_kwargs={"k": 3})
retrieved_docs = retriever.invoke("Hi there")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"{i}. {doc.page_content}")


1. Hi there!
2. Hi there!
3. Hi there!


### Document Embeddings (PGVector.from_documents)

In [19]:
from langchain_community.document_loaders import TextLoader

raw_documents = TextLoader('./llm_notes.txt', encoding="utf-8").load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(raw_documents)

In [20]:
db = PGVector.from_documents(   documents,
                                databricks_embeddings,
                                collection_name="llm_notes",
                                connection=pgvectordb_conn,
                                use_jsonb=True,
                                pre_delete_collection=True)


In [21]:
enable_logging()
db.similarity_search("softamx", k=4)
disable_logging()

DEBUG:openai._base_client:Request options: {'method': 'post', 'url': '/embeddings', 'files': None, 'idempotency_key': 'stainless-python-retry-c1026254-fb69-4bd2-bb0a-3cfb2b20211f', 'post_parser': <function Embeddings.create.<locals>.parser at 0x000001DD203238A0>, 'security': {'bearer_auth': True}, 'content': None, 'json_data': {'input': ['softamx'], 'model': 'databricks-gte-large-en', 'encoding_format': 'base64'}}
DEBUG:openai._base_client:Sending HTTP Request: POST https://adb-2907158998162760.0.azuredatabricks.net//serving-endpoints/embeddings
DEBUG:httpcore.connection:close.started
DEBUG:httpcore.connection:close.complete
DEBUG:httpcore.connection:connect_tcp.started host='adb-2907158998162760.0.azuredatabricks.net' port=443 local_address=None timeout=None socket_options=None
DEBUG:httpcore.connection:connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x000001DD1EA26A30>
DEBUG:httpcore.connection:start_tls.started ssl_context=<ssl.SSLContext object at 0x

### Add New Documents

In [22]:
import uuid
from langchain_core.documents import Document

# add_documents: adds documents to the vector store, embedding them in the process.
# This is useful for adding new documents to an existing collection. 
# You can also specify your own document IDs, as shown below.
enable_logging()

ids = [str(uuid.uuid4()), str(uuid.uuid4())]
db.add_documents(
    [
        Document(
            page_content="there are cats in the pond",
            metadata={"location": "pond", "topic": "animals"},
        ),
        Document(
            page_content="ducks are also found in the pond",
            metadata={"location": "pond", "topic": "animals"},
        ),
    ],
    ids=ids,
)

disable_logging()


DEBUG:openai._base_client:Request options: {'method': 'post', 'url': '/embeddings', 'files': None, 'idempotency_key': 'stainless-python-retry-f94eb5e0-6ee8-45b5-accd-c66430a0ee04', 'post_parser': <function Embeddings.create.<locals>.parser at 0x000001DD20323A00>, 'security': {'bearer_auth': True}, 'content': None, 'json_data': {'input': ['there are cats in the pond', 'ducks are also found in the pond'], 'model': 'databricks-gte-large-en', 'encoding_format': 'base64'}}
DEBUG:openai._base_client:Sending HTTP Request: POST https://adb-2907158998162760.0.azuredatabricks.net//serving-endpoints/embeddings
DEBUG:httpcore.connection:close.started
DEBUG:httpcore.connection:close.complete
DEBUG:httpcore.connection:connect_tcp.started host='adb-2907158998162760.0.azuredatabricks.net' port=443 local_address=None timeout=None socket_options=None
DEBUG:httpcore.connection:connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x000001DD1EB3CE50>
DEBUG:httpcore.connection:sta